In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import syllables
import torch

from features import AlphabetDataset  # noqa: F401

RESOURCES = Path("res")

In [2]:
dataset = torch.load(RESOURCES / "data.pt", weights_only=False)

In [3]:
stack = torch.load(RESOURCES / "results.pt")
df = pd.DataFrame(
    {
        "word": dataset.words,
        "prob": stack[
            torch.arange(len(stack)), [label for _, label, _ in dataset]
        ].detach(),
        "pred": [chr(label + ord("a")) for label in stack.argmax(dim=-1)],
        "label": dataset.labels,
        "syllables": [syllables.estimate(word) for word in dataset.words],
        "length": [len(word) for word in dataset.words],
    }
)
# ignore words that are too long
indices = (df["syllables"] <= 3) & (df["length"] < 10)
stack = stack[indices]
df = df[indices].reset_index(drop=True)
df.head()

,word,prob,pred,label,syllables,length
0,a,0.955523,a,a,1,1
1,aa,0.986235,a,a,1,2
2,aaliyah,0.882430,a,a,2,7
3,aaron,0.170024,e,a,2,5
4,ab,0.577771,a,a,1,2


In [4]:
indices = []
for i in range(26):
    mask = (df["label"] == chr(ord("a") + i)).to_numpy()
    (values,) = np.where(mask)
    indices.append(values[int(stack[mask, i].argmax().item())])
new_words = df.loc[indices].reset_index(drop=True)
new_words

,word,prob,pred,label,syllables,length
0,added,0.998603,a,a,2,5
1,barker,0.999953,b,b,2,6
2,clauses,0.996528,c,c,2,7
3,defer,0.999951,d,d,2,5
4,emerald,0.996770,e,e,3,7
5,front,0.991971,f,f,1,5
6,gained,0.998595,g,g,2,6
7,hand,0.999063,h,h,1,4
8,isabel,0.997028,i,i,3,6
9,jute,0.982760,j,j,1,4


In [5]:
indices = []
for i in range(26):
    mask = (
        (df["label"] == chr(ord("a") + i))
        # ignore c/k confusion (leads to less interesting results without)
        & ~((df["label"] == "k") & (df["pred"] == "c"))
    ).to_numpy()
    (values,) = np.where(mask)
    indices.append(values[int(stack[mask, i].argmin().item())])
worst_words = df.loc[indices].reset_index(drop=True)
worst_words

,word,prob,pred,label,syllables,length
0,aegis,2.668887e-04,e,a,2,5
1,bdellium,3.041433e-07,d,b,2,8
2,czar,1.013557e-03,z,c,1,4
3,djibouti,7.832997e-02,g,d,3,8
4,eunuch,2.097358e-03,u,e,2,6
5,fillet,3.523364e-01,p,f,2,6
6,gnostic,2.226065e-03,n,g,2,7
7,heir,7.926630e-05,e,h,1,4
8,ignoble,1.967931e-03,e,i,3,7
9,jaeger,1.305542e-03,y,j,2,6
